# Creación de una Base de Datos Analítica

## Integrantes

## Problemática Abordada

### La empresa AutoMercado de compra y venta de vehículos usados necesita contar con una base de datos que centralice la información de los carros disponibles para la venta. Actualmente los datos se encuentran dispersos y sin estructura, lo que dificulta comparar precios, identificar disponibilidad, conocer el historial del vehículo y analizar las preferencias del mercado.
### Al organizar los datos en una base estructurada, la empresa podrá consultar rápidamente vehículos según características específicas (año, transmisión, combustible, precio), evaluar tendencias de demanda y tomar decisiones de compra y venta basadas en datos, mejorando así la eficiencia comercial y la competitividad en el mercado.

## Objetivo

### Desarrollar una base de datos estructurada para AutoMercado Select que permita almacenar, consultar y analizar la información de los vehículos usados en inventario, facilitando la comparación de precios, el seguimiento de disponibilidad y el soporte a decisiones comerciales basadas en datos

## Dataset Escogido

Nombre: Datos de coches usados ​​en sitios web

Fuente: kaggle

Enlace: https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho?select=CAR+DETAILS+FROM+CAR+DEKHO.csv

Autor: Nehal Birla · Nishant Verma · Nikhil Kushwaha

Escogimos este dataset porque contiene información real del mercado de vehículos usados, permitiendo analizar cómo influyen factores como la marca, modelo, año, kilometraje, tipo de combustible y transmisión en el precio de venta. Además, ofrece datos variados y suficientemente completos para construir un modelo entidad-relación, normalizar la información y realizar consultas analíticas que apoyen la toma de decisiones en un contexto de compra y venta de autos.

### Variables Relevantes

| **Variable (ES)** | **Variable (EN)** | **Tipo**   | **Descripción**                                                  |
| ----------------- | ----------------- | ---------- | ---------------------------------------------------------------- |
| **Make**          | Brand             | Categórica | Marca del vehículo.                                              |
| **Model**         | Model             | Categórica | Modelo específico del vehículo.                                  |
| **Year**          | Year              | Numérica   | Año de fabricación del vehículo.                                 |
| **Price**         | Price             | Numérica   | Precio de venta actual del vehículo.                             |
| **Kilometer**     | Mileage           | Numérica   | Kilometraje recorrido por el vehículo (nivel de uso y desgaste). |
| **Fuel_Type**     | Fuel Type         | Categórica | Tipo de combustible utilizado.                                   |
| **Transmission**  | Transmission      | Categórica | Tipo de transmisión (Manual o Automática).                       |
| **Owner**         | Previous Owners   | Categórica | Cantidad de dueños anteriores.                                   |
| **Seller_Type**   | Seller Type       | Categórica | Tipo de vendedor (Particular o Concesionario).                   |


### Modelo Entidad-Relación

### Cargar el CSV

In [1]:
!pip install kagglehub[pandas-datasets]>=0.3.8


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importación de Librerias

In [2]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub

ModuleNotFoundError: No module named 'matplotlib'

### Funciones para Descargar, Extraer y el Leer el Dataset

In [0]:
def download_dataset_zip(url = ""):
        print("Descargando dataset desde Kaggle...")
        dataset_path = kagglehub.dataset_download(url)
        print("Ruta al dataset:", dataset_path)
        return dataset_path

def extract_zip_files(dataset_path):
        zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
        if zip_files:
            zip_file = os.path.join(dataset_path, zip_files[0])
            extract_dir = os.path.join(dataset_path, "extracted")
            os.makedirs(extract_dir, exist_ok=True)
            print(f"Extrayendo {zip_file} en {extract_dir}...")
            with zipfile.ZipFile(zip_file, "r") as z:
                z.extractall(extract_dir)
            return extract_dir
        else:
            # Si no se encuentra un ZIP, se verifica si existen archivos CSV en la ruta
            csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
            if csv_files:
                print("No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.")
                return dataset_path
            else:
                raise FileNotFoundError("No se encontró ningún archivo .zip ni archivos .csv en la ruta del dataset")

def create_csv(csv_dir):
        #os.makedirs('src/static/csv', exist_ok=True)
        csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No se encontraron archivos CSV en el directorio extraído")

        for file in csv_files:
            file_path = os.path.join(csv_dir, file)
            print(f"Leyendo {file_path}...")
            try:
                df = pd.read_csv(file_path, encoding="latin1")
            except Exception as e:
                print(f"Error al leer {file}: {e}")
                continue
            print(f"Creando/actualizando ")
        print("cvs creado correctamente en ")
        return df

### Descarga del Dataset

In [0]:
# 1. Descargar el dataset desde Kaggle
dataset_path = download_dataset_zip("nehalbirla/vehicle-dataset-from-cardekho")

# 2. Extraer los archivos (o detectar CSV)
csv_dir = extract_zip_files(dataset_path)

# 3. Crear el DataFrame de Pandas
df_pd = create_csv(csv_dir)

# 4. Mostrar las primeras filas
df_pd.head(4)


Descargando dataset desde Kaggle...
Ruta al dataset: /home/spark-6dabc6c5-eb35-4ac4-ae8e-3e/.cache/kagglehub/datasets/nehalbirla/vehicle-dataset-from-cardekho/versions/4
No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.
Leyendo /home/spark-6dabc6c5-eb35-4ac4-ae8e-3e/.cache/kagglehub/datasets/nehalbirla/vehicle-dataset-from-cardekho/versions/4/car data.csv...
Creando/actualizando 
Leyendo /home/spark-6dabc6c5-eb35-4ac4-ae8e-3e/.cache/kagglehub/datasets/nehalbirla/vehicle-dataset-from-cardekho/versions/4/Car details v3.csv...
Creando/actualizando 
Leyendo /home/spark-6dabc6c5-eb35-4ac4-ae8e-3e/.cache/kagglehub/datasets/nehalbirla/vehicle-dataset-from-cardekho/versions/4/CAR DETAILS FROM CAR DEKHO.csv...
Creando/actualizando 
Leyendo /home/spark-6dabc6c5-eb35-4ac4-ae8e-3e/.cache/kagglehub/datasets/nehalbirla/vehicle-dataset-from-cardekho/versions/4/car details v4.csv...
Creando/actualizando 
cvs creado correctamente en 


,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0


### Convertir Dataframe de pandas a Spark

In [0]:
df = spark.createDataFrame(df_pd)

### Creación la Tabla e Inserto de Datos

In [0]:
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

df.write.mode("overwrite").saveAsTable("tbl_ventas_carros")

### Verifica la creación correcta de la tabla

In [0]:
%sql
SELECT * 
FROM tbl_ventas_carros
LIMIT 5;
     

Make,Model,Price,Year,Kilometer,Fuel_Type,Transmission,Location,Color,Owner,Seller_Type,Engine,Max_Power,Max_Torque,Drivetrain,Length,Width,Height,Seating_Capacity,Fuel_Tank_Capacity
Maruti Suzuki,Alto 800 Lxi CNG,265000,2017,30000,CNG,Manual,Kanpur,White,First,Individual,796 cc,47 bhp @ 6000 rpm,69 Nm @ 3500 rpm,FWD,3395.0,1490.0,1475.0,5.0,35.0
Tata,Tiago Revotron XE [2016-2019],395000,2017,46992,Petrol,Manual,Gurgaon,White,First,Individual,1199 cc,84 bhp @ 6000 rpm,114 Nm @ 3500 rpm,FWD,3746.0,1647.0,1535.0,5.0,35.0
Honda,Jazz V Petrol,500000,2016,54440,Petrol,Manual,Mumbai,Silver,First,Individual,1199 cc,89 bhp @ 6000 rpm,110 Nm @ 4800 rpm,FWD,3955.0,1694.0,1544.0,5.0,40.0
Mahindra,Alturas G4 4WD AT [2018-2020],2770000,2019,32000,Diesel,Automatic,Faridabad,Black,First,Individual,2157 cc,178 bhp @ 4000 rpm,420 Nm @ 1600 rpm,AWD,4850.0,1960.0,1845.0,7.0,70.0
Mercedes-Benz,M-Class ML 250 CDI,2450000,2015,103000,Diesel,Automatic,Mumbai,White,First,Individual,2143 cc,203 bhp @ 4200 rpm,500 Nm @ 1600 rpm,AWD,4804.0,2141.0,1796.0,5.0,70.0


### Conteo de Registros

In [0]:
%sql
SELECT COUNT(*) FROM tbl_ventas_carros;

COUNT(*)
2059


### Nombres y Tipos de Columnas

In [0]:

%sql
DESCRIBE TABLE tbl_ventas_carros;

col_name,data_type,comment
Make,string,null
Model,string,null
Price,bigint,null
Year,bigint,null
Kilometer,bigint,null
Fuel_Type,string,null
Transmission,string,null
Location,string,null
Color,string,null
Owner,string,null


Esta consulta muestra la estructura de la tabla tbl_ventas_carros, es decir, lista cada columna junto con su tipo de dato y si permite valores nulos. Sirve para conocer cómo está organizada la información dentro de la tabla.

### Consulta con Filtro

In [0]:
%sql
SELECT Make, Model, Year, Price, Fuel_Type
FROM tbl_ventas_carros
WHERE Fuel_Type = 'Petrol'
ORDER BY Price ASC
LIMIT 10;



Make,Model,Year,Price,Fuel_Type
Tata,Nano Base,2010,49000,Petrol
Maruti Suzuki,Zen LXi BS-II,2004,71001,Petrol
Honda,City 1.5 EXi,2002,100000,Petrol
Tata,Manza Aqua Safire BS-IV,2012,114999,Petrol
Hyundai,i10 Era,2009,130000,Petrol
Maruti Suzuki,Estilo VXi,2008,135000,Petrol
Maruti Suzuki,Alto LXi BS-III,2009,140000,Petrol
Hyundai,Santro GLS,2009,141000,Petrol
Maruti Suzuki,Wagon R LXi Minor,2009,145000,Petrol
Maruti Suzuki,Alto LXi BS-III,2011,150000,Petrol


Esta consulta selecciona los vehículos que funcionan con gasolina (Fuel_Type = 'Petrol') y muestra sus datos básicos: marca, modelo, año y precio. Luego los ordena de menor a mayor precio (ORDER BY Price ASC) y devuelve solo los 10 vehículos más económicos (LIMIT 10).

### Creacion de la base de datos

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS automercado_select;
USE automercado_select;

CREATE TABLE IF NOT EXISTS marcas (
    id_marca BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nombre_marca STRING NOT NULL
);

CREATE TABLE IF NOT EXISTS modelos (
    id_modelo BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nombre_modelo STRING NOT NULL,
    id_marca BIGINT NOT NULL,
    FOREIGN KEY (id_marca) REFERENCES marcas(id_marca)
);

CREATE TABLE IF NOT EXISTS combustibles (
    id_combustible BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    tipo_combustible STRING NOT NULL
);

CREATE TABLE IF NOT EXISTS transmisiones (
    id_transmision BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    tipo_transmision STRING NOT NULL
);

CREATE TABLE IF NOT EXISTS vendedores (
    id_vendedor BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    tipo_vendedor STRING NOT NULL  
);

CREATE TABLE IF NOT EXISTS carros (
    id_carro BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    anio INT,
    kilometraje BIGINT,
    color STRING,
    precio BIGINT,
    motor STRING,
    potencia_max STRING,
    torque_max STRING,
    capacidad_tanque DOUBLE,
    asientos DOUBLE,
    largo DOUBLE,
    ancho DOUBLE,
    alto DOUBLE,
    id_modelo BIGINT NOT NULL,
    id_combustible BIGINT NOT NULL,
    id_transmision BIGINT NOT NULL,
    id_vendedor BIGINT NOT NULL,
    FOREIGN KEY (id_modelo) REFERENCES modelos(id_modelo),
    FOREIGN KEY (id_combustible) REFERENCES combustibles(id_combustible),
    FOREIGN KEY (id_transmision) REFERENCES transmisiones(id_transmision),
    FOREIGN KEY (id_vendedor) REFERENCES vendedores(id_vendedor)
);


La base de datos AutoMercado Select permite organizar y administrar de manera estructurada la información relacionada con la venta de vehículos usados. A través de su modelo entidad–relación, se logra separar adecuadamente los datos del fabricante, modelos, características y transacciones, lo cual facilita el análisis posterior. Con esta estructura, es posible consultar tendencias de precios, preferencias de combustible, variaciones entre marcas y el comportamiento del mercado en general. La organización clara y normalizada de los datos contribuye a mejorar la calidad de las decisiones comerciales, permitiendo identificar patrones relevantes y apoyar estrategias de venta y adquisición de vehículos dentro del negocio.